# Cognee Exercise Knowledge Graph

A class-based refactor: each piece (token/latency tracking, graph storage, question
routing, context retrieval, graph visualization, golden-set evaluation) is its own
class so this can be lifted into a service/module rather than run top-to-bottom as
notebook scratch code.

**Architecture**
1. `TokenUsageTracker` — wraps litellm to track call latency + token usage by phase.
2. `ExerciseGraphBuilder` — builds deduplicated `DataPoint` nodes from raw exercise JSON.
3. `CogneeExerciseStore` — pushes nodes into Cognee's graph + vector indexes.
4. `QuestionRouter` — classifies a question (local / instruction / multi_hop / global) and
   maps it to the best-fit `SearchType`.
5. `GraphContextRetriever` — fetches the raw graph triplets relevant to a question
   (via `GraphCompletionRetriever`, used directly since it reliably returns real `Edge` objects).
6. `GraphVisualizer` — renders a searchable, self-contained HTML graph (no CDN dependency).
7. `CogneeQAEngine` — orchestrates 1+4+5+6 into a single `.ask(question)` call.
8. `QAEvaluator` — runs a golden Q&A set across multiple `SearchType`s and exports an Excel report.

In [ ]:
!pip install -q python-dotenv pyvis openpyxl

In [2]:
print("hi")


hi


In [3]:
import os, pathlib, time, json
from dotenv import load_dotenv

load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("LLM_API_KEY") or ""
if not GEMINI_API_KEY:
    raise EnvironmentError("Set GEMINI_API_KEY (or LLM_API_KEY) in your .env file.")

os.environ.setdefault("LLM_API_KEY",            GEMINI_API_KEY)
os.environ.setdefault("LLM_PROVIDER",           "gemini")
os.environ.setdefault("LLM_MODEL",              "gemini/gemini-2.5-flash-lite")
os.environ.setdefault("EMBEDDING_API_KEY",      GEMINI_API_KEY)
os.environ.setdefault("EMBEDDING_PROVIDER",     "gemini")
os.environ.setdefault("EMBEDDING_MODEL",        "gemini/gemini-embedding-001")
os.environ.setdefault("EMBEDDING_DIMENSIONS",   "768")
os.environ.setdefault("GRAPH_DATABASE_PROVIDER","neo4j")
os.environ.setdefault("GRAPH_DATABASE_NAME",    "neo4j")
os.environ["ENABLE_BACKEND_ACCESS_CONTROL"]     = "false"
os.environ["COGNEE_SKIP_CONNECTION_TEST"]       = "true"

print(f"API key: {GEMINI_API_KEY[:6]}… | LLM: {os.environ['LLM_MODEL']}")

API key: AQ.Ab8… | LLM: gemini/gemini-2.5-flash-lite


## Step 1 — Cognee setup

In [4]:
import cognee
from cognee import config, prune, search, SearchType, visualize_graph
from cognee.low_level import setup, DataPoint
from cognee.pipelines import run_tasks, Task
from cognee.tasks.storage import add_data_points
from cognee.tasks.storage.index_graph_edges import index_graph_edges
from cognee.modules.users.methods import get_default_user
from cognee.modules.data.methods import load_or_create_datasets
from cognee.infrastructure.engine.models.Edge import Edge
from typing import List, Optional, Tuple



2026-06-22T09:36:08.773236 [info     ] Deleted old log file: C:\Users\rageshwari.swamy\AppData\Local\anaconda3\envs\cognee-2\Lib\site-packages\logs\2026-06-22_12-43-48.log [cognee.shared.logging_utils]

2026-06-22T09:36:08.776409 [info     ] Deleted old log file: C:\Users\rageshwari.swamy\AppData\Local\anaconda3\envs\cognee-2\Lib\site-packages\logs\2026-06-22_09-38-26.log [cognee.shared.logging_utils]

2026-06-22T09:36:20.376287 [warning  ] From version 0.5.0 onwards, Cognee will run with multi-user access control mode set to on by default. Data isolation between different users and datasets will be enforced and data created before multi-user access control mode was turned on won't be accessible by default. To disable multi-user access control mode and regain access to old data set the environment variable ENABLE_BACKEND_ACCESS_CONTROL to false before starting Cognee. For more information, please refer to the Cognee documentation. [cognee.shared.logging_utils]

2026-06-22T09:36:20.379

In [5]:
config.data_root_directory(str(pathlib.Path("./.cognee_data").resolve()))
config.system_root_directory(str(pathlib.Path("./.cognee_system").resolve()))

await prune.prune_data()
await prune.prune_system(metadata=True)
await setup()
print("Cognee ready — clean slate.")


2026-06-22T09:37:05.824724 [warning  ] From version 0.5.0 onwards, Cognee will run with multi-user access control mode set to on by default. Data isolation between different users and datasets will be enforced and data created before multi-user access control mode was turned on won't be accessible by default. To disable multi-user access control mode and regain access to old data set the environment variable ENABLE_BACKEND_ACCESS_CONTROL to false before starting Cognee. For more information, please refer to the Cognee documentation. [cognee.shared.logging_utils]

2026-06-22T09:37:05.830006 [info     ] Log file created at: C:\Users\rageshwari.swamy\AppData\Local\anaconda3\envs\cognee-2\Lib\site-packages\logs\2026-06-22_15-06-07.log [cognee.shared.logging_utils] log_file=C:\Users\rageshwari.swamy\AppData\Local\anaconda3\envs\cognee-2\Lib\site-packages\logs\2026-06-22_15-06-07.log

2026-06-22T09:37:05.832611 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_

Cognee ready — clean slate.


## `TokenUsageTracker`

Wraps litellm's `completion` / `acompletion` / `embedding` / `aembedding` entrypoints
(monkeypatching, not `success_callback` — more reliable since callbacks can get reset
by libraries that configure litellm internally) to record latency + token usage per
logical **phase**.

**Production note:** instantiate once per process (singleton). `install()` guards
against double-patching if called twice.

In [6]:
from contextlib import contextmanager


class TokenUsageTracker:
    """Tracks LLM/embedding call latency and token usage by logical 'phase'."""

    def __init__(self):
        self.log: list[dict] = []
        self._current_phase = "unassigned"
        self._installed = False
        self._originals = {}

    def install(self):
        import litellm
        if self._installed:
            return
        self._originals = {
            "completion":  litellm.completion,
            "acompletion": litellm.acompletion,
            "embedding":   litellm.embedding,
            "aembedding":  litellm.aembedding,
        }
        orig = self._originals

        def _extract_usage(response):
            usage = getattr(response, "usage", None)
            if usage is None and isinstance(response, dict):
                usage = response.get("usage")
            if usage is None:
                return 0, 0, 0
            getter = usage.get if isinstance(usage, dict) else (lambda k, d=0: getattr(usage, k, d))
            p = getter("prompt_tokens", 0) or 0
            c = getter("completion_tokens", 0) or 0
            t = getter("total_tokens", 0) or (p + c)
            return p, c, t

        def _record(model, call_type, response, elapsed):
            p, c, t = _extract_usage(response)
            self.log.append({
                "phase": self._current_phase, "model": model, "call_type": call_type,
                "prompt_tokens": p, "completion_tokens": c, "total_tokens": t,
                "elapsed_sec": elapsed,
            })

        def patched_completion(*args, **kwargs):
            t0 = time.perf_counter()
            resp = orig["completion"](*args, **kwargs)
            _record(kwargs.get("model", "unknown"), "completion", resp, time.perf_counter() - t0)
            return resp

        async def patched_acompletion(*args, **kwargs):
            t0 = time.perf_counter()
            resp = await orig["acompletion"](*args, **kwargs)
            _record(kwargs.get("model", "unknown"), "completion", resp, time.perf_counter() - t0)
            return resp

        def patched_embedding(*args, **kwargs):
            t0 = time.perf_counter()
            resp = orig["embedding"](*args, **kwargs)
            _record(kwargs.get("model", "unknown"), "embedding", resp, time.perf_counter() - t0)
            return resp

        async def patched_aembedding(*args, **kwargs):
            t0 = time.perf_counter()
            resp = await orig["aembedding"](*args, **kwargs)
            _record(kwargs.get("model", "unknown"), "embedding", resp, time.perf_counter() - t0)
            return resp

        litellm.completion  = patched_completion
        litellm.acompletion = patched_acompletion
        litellm.embedding   = patched_embedding
        litellm.aembedding  = patched_aembedding
        self._installed = True

    def uninstall(self):
        import litellm
        if not self._installed:
            return
        litellm.completion  = self._originals["completion"]
        litellm.acompletion = self._originals["acompletion"]
        litellm.embedding   = self._originals["embedding"]
        litellm.aembedding  = self._originals["aembedding"]
        self._installed = False

    def set_phase(self, name: str):
        self._current_phase = name

    @contextmanager
    def phase(self, name: str):
        previous = self._current_phase
        self.set_phase(name)
        try:
            yield
        finally:
            self._current_phase = previous

    def summarize(self, name: str) -> dict:
        entries = [e for e in self.log if e["phase"] == name]
        return {
            "calls": len(entries),
            "prompt_tokens": sum(e["prompt_tokens"] for e in entries),
            "completion_tokens": sum(e["completion_tokens"] for e in entries),
            "total_tokens": sum(e["total_tokens"] for e in entries),
            "llm_time_sec": sum(e["elapsed_sec"] for e in entries if e["elapsed_sec"]),
        }

    def as_dataframe(self):
        import pandas as pd
        return pd.DataFrame(self.log)


tracker = TokenUsageTracker()
tracker.install()
print("Token/time tracking enabled (TokenUsageTracker.install())")

Token/time tracking enabled (TokenUsageTracker.install())


### Pydantic DataPoint models

Each node type maps to a Neo4j label. `Exercise` has typed edges to `Level`,
`Category`, `Equipment`, `Muscle` (primary target), and `SecondaryMuscle`.

```
Exercise ──[HAS_LEVEL]─────────► Level
         ──[IN_CATEGORY]─────────► Category
         ──[REQUIRES]───────────► Equipment
         ──[HAS_PRIMARY_MUSCLE]──► Muscle
         ──[HAS_SECONDARY_MUSCLE]► SecondaryMuscle  (list edge)
```

In [7]:
class Muscle(DataPoint):
    """A muscle or muscle group, e.g. 'abs', 'triceps', 'lats'."""
    name: str
    metadata: dict = {"index_fields": ["name"]}

class SecondaryMuscle(DataPoint):
    """A secondary muscle that is worked by an exercise, e.g. 'hip flexors', 'lower back'."""
    name: str
    metadata: dict = {"index_fields": ["name"]}


class Equipment(DataPoint):
    """Equipment used to perform an exercise, e.g. 'body weight', 'cable'."""
    name: str
    metadata: dict = {"index_fields": ["name"]}


class Level(DataPoint):
    """Difficulty level of an exercise, e.g. 'Beginner', 'Intermediate', 'Hard'."""
    name: str
    metadata: dict = {"index_fields": ["name"]}


class Category(DataPoint):
    """Exercise category, e.g. 'waist', 'back', 'chest'. Often matches body_part
    but is kept as its own node type since the source data treats them as
    distinct fields."""
    name: str
    metadata: dict = {"index_fields": ["name"]}

class Target(DataPoint):
    """The primary target muscle of an exercise, e.g. 'abs', 'lats', 'pectorals'."""
    name: str
    metadata: dict = {"index_fields": ["name"]}


class Exercise(DataPoint):
    """A single exercise, with edges to its level, category, body part,
    equipment, and target/secondary muscles."""
    exercise_id: str
    name: str
    instructions: str

    has_level: Tuple[Edge, Level]
    in_category: Tuple[Edge, Category]
    requires: Tuple[Edge, Equipment]
    has_primary_muscle: Tuple[Edge, Muscle]
    has_secondary_muscle: Tuple[Edge, List[SecondaryMuscle]] = (Edge(relationship_type="HAS_SECONDARY_MUSCLE"), [])
    targets: Tuple[Edge, Target]

    metadata: dict = {
        "index_fields": ["name", "instructions"]
    }

### `ExerciseGraphBuilder`

Builds deduplicated `Exercise` nodes (+ shared `Level`/`Category`/`Equipment`/`Muscle`/
`Target`/`SecondaryMuscle` nodes, so identical names map to the same graph node instead
of duplicates) from the raw exercise JSON.

In [8]:
class ExerciseGraphBuilder:
    """Builds deduplicated Exercise DataPoint nodes from raw exercise JSON records."""

    def __init__(self):
        self._muscle_cache: dict[str, Muscle] = {}
        self._target_cache: dict[str, Target] = {}
        self._equipment_cache: dict[str, Equipment] = {}
        self._level_cache: dict[str, Level] = {}
        self._category_cache: dict[str, Category] = {}
        self._secondary_muscle_cache: dict[str, SecondaryMuscle] = {}

    @staticmethod
    def load_exercises(file_path: str) -> list[dict]:
        with open(file_path, "r") as f:
            return json.load(f)

    def _get_or_create(self, cache: dict, cls, name: str):
        name = name.strip().lower()
        if name not in cache:
            cache[name] = cls(name=name)
        return cache[name]

    def get_target(self, name: str) -> Target:
        return self._get_or_create(self._target_cache, Target, name)

    def get_muscle(self, name: str) -> Muscle:
        return self._get_or_create(self._muscle_cache, Muscle, name)

    def get_equipment(self, name: str) -> Equipment:
        return self._get_or_create(self._equipment_cache, Equipment, name)

    def get_level(self, name: str) -> Level:
        return self._get_or_create(self._level_cache, Level, name)

    def get_category(self, name: str) -> Category:
        return self._get_or_create(self._category_cache, Category, name)

    def get_secondary_muscle(self, name: str) -> SecondaryMuscle:
        return self._get_or_create(self._secondary_muscle_cache, SecondaryMuscle, name)

    def build_exercise_node(self, ex: dict) -> Exercise:
        secondary = [self.get_secondary_muscle(n) for n in ex.get("secondary_muscles", [])]
        return Exercise(
            exercise_id=ex["id"],
            name=ex["name"],
            instructions=ex["instructions"],
            has_level=(Edge(edge_text="which level it belongs to"), self.get_level(ex["level"])),
            in_category=(Edge(edge_text="which body part it belongs to"), self.get_category(ex["category"])),
            requires=(Edge(edge_text="which equipment it requires"), self.get_equipment(ex["equipment"])),
            targets=(Edge(edge_text="which muscle it targets"), self.get_target(ex["target"])),
            has_primary_muscle=(Edge(edge_text="which primary muscle it works"), self.get_muscle(ex["muscle_group"])),
            has_secondary_muscle=(Edge(edge_text="which secondary muscles it works"), secondary),
        )

    def build_all(self, exercises: list[dict]) -> list[Exercise]:
        seen_ids = set()
        nodes = []
        for ex in exercises:
            if ex["id"] in seen_ids:
                continue
            seen_ids.add(ex["id"])
            nodes.append(self.build_exercise_node(ex))
        return nodes

    def summary(self) -> dict:
        return {
            "levels": list(self._level_cache),
            "categories": list(self._category_cache),
            "equipment": list(self._equipment_cache),
            "muscles": list(self._muscle_cache),
            "targets": list(self._target_cache),
        }

In [9]:
EXERCISES_PATH = "D:\longmemory\LLM-Memory\exercises_500_transformed.json"  # adjust to your environment

EXERCISES = ExerciseGraphBuilder.load_exercises(EXERCISES_PATH)
print(f"Exercises to store: {len(EXERCISES)}")

graph_builder = ExerciseGraphBuilder()
exercise_nodes = graph_builder.build_all(EXERCISES)

print(f"Built {len(exercise_nodes)} Exercise nodes")
for key, values in graph_builder.summary().items():
    print(f"Unique {key}: {values}")

Exercises to store: 1
Built 1 Exercise nodes
Unique levels: ['beginner']
Unique categories: ['cardio']
Unique equipment: ['body weight']
Unique muscles: ['quadriceps']
Unique targets: ['cardiovascular system']


<>:1: SyntaxWarning: invalid escape sequence '\l'
<>:1: SyntaxWarning: invalid escape sequence '\l'
C:\Users\rageshwari.swamy\AppData\Local\Temp\ipykernel_8492\821717808.py:1: SyntaxWarning: invalid escape sequence '\l'
  EXERCISES_PATH = "D:\longmemory\LLM-Memory\exercises_500_transformed.json"  # adjust to your environment


In [10]:
async def add_exercise_data_points(data, ctx=None):
    return await add_data_points(data, embed_triplets=True)

In [12]:
class CogneeExerciseStore:
    """Wraps the Cognee BYOG (bring-your-own-graph) ingestion pipeline: pushes
    pre-built DataPoint nodes into the graph + vector indexes, with timing/token
    tracking via the shared TokenUsageTracker."""

    def __init__(self, dataset_name: str, tracker: TokenUsageTracker):
        self.dataset_name = dataset_name
        self.tracker = tracker
        self.dataset_id = None
        self.last_storage_stats: Optional[dict] = None

    async def push(self, nodes: list) -> dict:
        user = await get_default_user()
        datasets = await load_or_create_datasets([self.dataset_name], [], user)
        self.dataset_id = datasets[0].id

        pipeline = run_tasks(
            [Task(add_exercise_data_points)],
            self.dataset_id,
            nodes,
            user,
            "exercises_byog_pipeline",
        )

        with self.tracker.phase("graph_and_embedding_creation"):
            t0 = time.perf_counter()
            async for status in pipeline:
                print("Pipeline status:", status)
            await index_graph_edges()
            elapsed = time.perf_counter() - t0

        self.last_storage_stats = {
            "elapsed_sec": elapsed,
            **self.tracker.summarize("graph_and_embedding_creation"),
        }
        return self.last_storage_stats

    async def visualize(self, path: str = "./.artifacts/exercises_graph.html") -> str:
        out_path = pathlib.Path(path).resolve()
        out_path.parent.mkdir(parents=True, exist_ok=True)
        await visualize_graph(str(out_path))
        return str(out_path)

In [13]:
store = CogneeExerciseStore(dataset_name="exercises_demo", tracker=tracker)
stats = await store.push(exercise_nodes)

print("Graph loaded into Cognee \u2705")
print(f"\n\u23f1  Storage time: {stats['elapsed_sec']:.2f}s")
print(f"\U0001f522 Calls: {stats['calls']} | Tokens: {stats['total_tokens']} "
      f"(prompt={stats['prompt_tokens']}, completion={stats['completion_tokens']})")


User 061bfe20-2f0d-4d6d-b397-9c923b8b93de has registered.

2026-06-22T09:37:59.863076 [info     ] Pipeline run started: `41a013fe-03af-5cb4-a221-2b037510eaac` [run_tasks_with_telemetry()]

2026-06-22T09:37:59.875515 [info     ] Coroutine task started: `add_exercise_data_points` [run_tasks_base]

2026-06-22T09:37:59.901710 [info     ] Completed graph extraction for DataPoint [cognee.shared.logging_utils] extra={'datapoint_id': 'd00f2047-f361-46a7-85b0-aae58d398e32', 'nodes_extracted': 1, 'edges_extracted': 0}

2026-06-22T09:37:59.910142 [info     ] Completed graph extraction for DataPoint [cognee.shared.logging_utils] extra={'datapoint_id': 'bd516a7a-5a32-4a80-99ae-4862e07860ac', 'nodes_extracted': 1, 'edges_extracted': 0}

2026-06-22T09:37:59.915928 [info     ] Completed graph extraction for DataPoint [cognee.shared.logging_utils] extra={'datapoint_id': 'c2281947-1e0f-4b04-bf66-741558328ba7', 'nodes_extracted': 1, 'edges_extracted': 0}

2026-06-22T09:37:59.929074 [info     ] Completed

Pipeline status: status='PipelineRunStarted' pipeline_run_id=UUID('5f115ee2-6ed3-5355-b0ba-dddbdcb3503a') dataset_id=UUID('b160f6ef-acc8-56ba-a93c-77600dc5f2fc') dataset_name='exercises_demo' payload=[Exercise(id=UUID('eb33a6a7-3ca1-4dd5-bc51-7198be6f77a1'), created_at=1782121064703, updated_at=1782121064703, ontology_valid=False, version=1, topological_rank=0, metadata={'index_fields': ['name', 'instructions']}, type='Exercise', belongs_to_set=None, source_pipeline=None, source_task=None, source_node_set=None, source_user=None, feedback_weight=0.5, exercise_id='3224', name='jack jump (male)', instructions='Stand with your feet together and your arms by your sides. Jump up, spreading your feet apart and raising your arms above your head. As you land, quickly jump back to the starting position. Repeat for the desired number of repetitions.', has_level=(Edge(weight=None, weights=None, relationship_type=None, properties=None, edge_text='which level it belongs to'), Level(id=UUID('d00f2047


2026-06-22T09:38:06.190904 [info     ] Created and indexed 7 triplets from graph structure [add_data_points]

2026-06-22T09:38:06.192981 [info     ] Coroutine task completed: `add_exercise_data_points` [run_tasks_base]

2026-06-22T09:38:06.200719 [info     ] Pipeline run completed: `41a013fe-03af-5cb4-a221-2b037510eaac` [run_tasks_with_telemetry()]


Pipeline status: status='PipelineRunCompleted' pipeline_run_id=UUID('5f115ee2-6ed3-5355-b0ba-dddbdcb3503a') dataset_id=UUID('b160f6ef-acc8-56ba-a93c-77600dc5f2fc') dataset_name='exercises_demo' payload=None data_ingestion_info=[{'run_info': PipelineRunCompleted(status='PipelineRunCompleted', pipeline_run_id=UUID('5f115ee2-6ed3-5355-b0ba-dddbdcb3503a'), dataset_id=UUID('b160f6ef-acc8-56ba-a93c-77600dc5f2fc'), dataset_name='exercises_demo', payload=None, data_ingestion_info=None)}]



2026-06-22T09:38:06.530409 [info     ] Retrieved 8 nodes and 7 edges in 0.18 seconds [Neo4jAdapter]

2026-06-22T09:38:06.531410 [warning  ] Your graph edge embedding is deprecated, please pass edges to the index_graph_edges directly. [cognee.shared.logging_utils]


Graph loaded into Cognee ✅

⏱  Storage time: 7.56s
🔢 Calls: 11 | Tokens: 579 (prompt=579, completion=0)


In [14]:
graph_file_path = await store.visualize()
print("Full graph visualization saved to:", graph_file_path)
print("Open this file in a browser to explore nodes & relationships interactively.")


2026-06-22T09:38:12.187009 [info     ] Retrieved 8 nodes and 7 edges in 0.08 seconds [Neo4jAdapter]

2026-06-22T09:38:12.192771 [info     ] Graph visualization saved as D:\longmemory\LLM-Memory\.artifacts\exercises_graph.html [cognee.shared.logging_utils]

2026-06-22T09:38:12.193807 [info     ] The HTML file has been stored at path: D:\longmemory\LLM-Memory\.artifacts\exercises_graph.html [cognee.shared.logging_utils]


Full graph visualization saved to: D:\longmemory\LLM-Memory\.artifacts\exercises_graph.html
Open this file in a browser to explore nodes & relationships interactively.


In [15]:
EXERCISE_QA_SYSTEM_PROMPT = """
You are a fitness knowledge assistant answering questions using ONLY the exercise graph context provided to you (exercise names, target muscles, secondary muscles, equipment, difficulty level, category, and instructions).

Rules:
1. Use the exact terminology found in the context (e.g. "pectorals", "glutes", "body weight", "beginner") rather than vague paraphrases.
2. Only mention equipment, muscles, or exercises that actually appear in the provided context. Never invent or assume ones that aren't present.
3. If the question implies a constraint (an injury, no equipment, a body part to avoid), make sure your answer respects that constraint based on what's in the context.
4. Be concise but always name the specific exercise(s) when relevant.
5. If the context doesn't contain enough information to answer, say so plainly instead of guessing.

Few-shot examples:

Example 1
Question: What equipment does the archer push up require?
Context: Exercise "archer push up" -> requires -> Equipment "body weight"; -> has_level -> Level "hard"; -> has_primary_muscle -> Muscle "pectorals"
Answer: The archer push up requires body weight only (no equipment) and targets the pectorals. It's rated hard difficulty.

Example 2
Question: I have a knee injury, what upper body exercises can I still do safely?
Context: Exercise "incline reverse grip push-up" -> in_category -> Category "chest" -> has_primary_muscle -> Muscle "pectorals"; Exercise "dumbbell reverse fly" -> in_category -> Category "shoulders" -> has_primary_muscle -> Muscle "delts"
Answer: Since these are upper body movements, the incline reverse grip push-up (chest, pectorals) and dumbbell reverse fly (shoulders, delts) don't load the knees, so they should be safe to try \u2014 but check with a professional given your injury.

Example 3
Question: What's a good exercise for leg day?
Context: (no leg-related exercises found in the retrieved context)
Answer: I don't have any leg-focused exercises in the retrieved context to recommend confidently \u2014 try rephrasing the question or confirm leg exercises were included in the dataset.
"""

In [16]:
def answer_to_str(results) -> str:
    """Flatten cognee.search() results to a single lowercase string."""
    if not results:
        return ""
    parts = []
    for r in results:
        if isinstance(r, str):
            parts.append(r)
        elif isinstance(r, dict):
            parts.append(" ".join(str(v) for v in r.values()))
        else:
            parts.append(str(r))
    return " ".join(parts).lower()

## Step 2 — Auto-routed QA engine

Splits questions into 4 scopes (local / instruction / multi_hop / global), routes each
to the best-fit `SearchType`, answers it, and retrieves + visualizes the graph context
that backed the answer — all wired together in `CogneeQAEngine`.

In [17]:
from pydantic import BaseModel
from typing import Literal
from cognee.infrastructure.llm.LLMGateway import LLMGateway


class QuestionScope(BaseModel):
    reasoning: str
    scope: Literal["local", "instruction", "multi_hop", "global"]


class QuestionRouter:
    """Classifies a fitness question into local / instruction / multi_hop / global
    and maps it to the cognee SearchType best suited for that scope."""

    SCOPE_CLASSIFIER_PROMPT = """
Classify the user's fitness-exercise question into exactly one scope:

- "local": about ONE specific exercise, muscle, or fact, NOT asking for step-by-step directions.
  e.g. "What equipment does the deadlift need?", "What muscle does the bench press target?"
- "instruction": explicitly asks HOW to perform an exercise, or for its steps/instructions/proper form.
  e.g. "Tell me the instructions for doing jumping jacks", "How do I do a hip bridge properly?"
- "multi_hop": combines two or more constraints that must be reasoned across together
  (an injury + a body region, an equipment limit + a goal, "X but not Y").
  e.g. "I have a knee injury, what upper body exercises can I still do safely?"
- "global": asks for a broad, dataset-wide view, summary, count, or category rather than one entity.
  Words like "overview", "categories", "how many", "all".
  e.g. "What categories of exercises exist?", "How many exercises are there?"

Give brief reasoning, then exactly one label.
"""

    SEARCH_TYPE_MAP = {
        "local":       SearchType.GRAPH_COMPLETION,
        "instruction": SearchType.TRIPLET_COMPLETION,
        "multi_hop":   SearchType.GRAPH_COMPLETION_COT,
        "global":      SearchType.GRAPH_SUMMARY_COMPLETION,
    }

    def __init__(self, tracker: TokenUsageTracker):
        self.tracker = tracker

    async def classify(self, question: str) -> QuestionScope:
        with self.tracker.phase("question_classification"):
            return await LLMGateway.acreate_structured_output(
                text_input=question,
                system_prompt=self.SCOPE_CLASSIFIER_PROMPT,
                response_model=QuestionScope,
            )

    def search_type_for(self, scope: str) -> SearchType:
        return self.SEARCH_TYPE_MAP[scope]

In [18]:
from cognee.modules.retrieval.graph_completion_retriever import GraphCompletionRetriever
from cognee.modules.graph.cognee_graph.CogneeGraphElements import Edge as CogneeEdge


class GraphContextRetriever:
    """Retrieves the raw graph triplets relevant to a question, independent of
    which SearchType is used to generate the final answer. Uses
    GraphCompletionRetriever directly (reliable \u2014 returns real Edge objects),
    rather than relying on search(..., verbose=True)'s objects_result, which is
    not populated consistently across search types."""

    def __init__(self, tracker: TokenUsageTracker, top_k: int = 5):
        self.tracker = tracker
        self.top_k = top_k

    @staticmethod
    def _node_label(node) -> str:
        attrs = node.attributes
        return attrs.get("name") or (attrs.get("text", "")[:40] + "...") or "Unnamed"

    @staticmethod
    def _edge_label(edge) -> str:
        return (
            edge.attributes.get("relationship_type")
            or edge.attributes.get("relationship_name")
            or edge.attributes.get("edge_text")
            or "related_to"
        )

    async def get_triples(self, question: str) -> list[tuple[str, str, str]]:
        with self.tracker.phase("context_retrieval"):
            retriever = GraphCompletionRetriever(top_k=self.top_k)
            edges = await retriever.get_retrieved_objects(query=question)

        triples = []
        for edge in edges or []:
            if not isinstance(edge, CogneeEdge):
                continue
            triples.append((
                self._node_label(edge.node1),
                self._edge_label(edge),
                self._node_label(edge.node2),
            ))
        return triples

In [20]:
class GraphVisualizer:
    """Renders (source, relation, target) triples as a self-contained, searchable
    HTML graph. Uses pyvis with cdn_resources="in_line" so the vis-network JS is
    embedded directly in the file \u2014 no CDN/internet dependency, which avoids the
    CSP/"untrusted output" issues that block remote <script src> tags in many
    notebook front-ends (e.g. VS Code's notebook webview)."""

    def __init__(self, output_dir: str = "./.artifacts/graphs"):
        self.output_dir = pathlib.Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def render(self, triples, title: str = "Graph context", filename: Optional[str] = None,
               height: int = 600, inline: bool = True) -> str:
        from pyvis.network import Network

        net = Network(height=f"{height}px", width="100%", directed=True,
                       notebook=True, cdn_resources="in_line")

        seen = set()
        for source, relation, target in triples:
            for n in (source, target):
                if n not in seen:
                    net.add_node(n, label=n)
                    seen.add(n)
            net.add_edge(source, target, label=relation)

        if not seen:
            print(f"{title}: no context retrieved.")
            return ""

        filename = filename or f"graph_{abs(hash(title))}.html"
        out_path = self.output_dir / filename
        net.write_html(str(out_path), open_browser=False, notebook=False)

        search_box = """
        <input type="text" id="nodeSearch" placeholder="Search node..."
               style="width:300px;padding:6px;margin:8px;">
        <script>
        document.getElementById('nodeSearch').addEventListener('input', function(e) {
            var q = e.target.value.toLowerCase();
            var matched = [];
            nodes.forEach(function(n) {
                var isMatch = q.length > 0 && n.label.toLowerCase().includes(q);
                nodes.update({id: n.id, color: isMatch ? '#ff5252' : '#97c2fc'});
                if (isMatch) matched.push(n.id);
            });
            if (matched.length > 0) { network.selectNodes(matched); network.fit({nodes: matched, animation: true}); }
        });
        </script>
        """
        html = out_path.read_text(encoding="utf-8")
        html = html.replace("</body>", search_box + "</body>", 1)
        out_path.write_text(html, encoding="utf-8")

        print(f"{title} -> saved to {out_path.resolve()}")
        if inline:
            from IPython.display import IFrame, display
            display(IFrame(str(out_path), width="100%", height=height + 60))
        return str(out_path)

In [21]:
class QAResult(BaseModel):
    question: str
    predicted_scope: str
    search_type_used: str
    answer: str
    context_triples: list[tuple[str, str, str]]
    classify_time_sec: float
    answer_time_sec: float
    classify_tokens: int
    answer_tokens: int


class CogneeQAEngine:
    """Production entry point: classifies a question, routes it to the best
    SearchType, generates the answer, and (optionally) retrieves + renders the
    graph context that backed the answer.

    Usage:
        engine = CogneeQAEngine(router, context_retriever, visualizer, tracker,
                                 system_prompt=EXERCISE_QA_SYSTEM_PROMPT)
        result = await engine.ask("How do I do a hip bridge?")
    """

    def __init__(self, router: QuestionRouter, context_retriever: GraphContextRetriever,
                 visualizer: GraphVisualizer, tracker: TokenUsageTracker,
                 system_prompt: Optional[str] = None):
        self.router = router
        self.context_retriever = context_retriever
        self.visualizer = visualizer
        self.tracker = tracker
        self.system_prompt = system_prompt

    async def ask(self, question: str, show_graph: bool = True) -> QAResult:
        t0 = time.perf_counter()
        classification = await self.router.classify(question)
        classify_elapsed = time.perf_counter() - t0
        classify_usage = self.tracker.summarize("question_classification")

        search_type = self.router.search_type_for(classification.scope)

        phase_name = f"auto_search::{search_type.value}"
        t1 = time.perf_counter()
        with self.tracker.phase(phase_name):
            raw_results = await search(query_text=question, query_type=search_type,
                                        system_prompt=self.system_prompt)
        answer_elapsed = time.perf_counter() - t1
        answer_usage = self.tracker.summarize(phase_name)

        answer = answer_to_str(raw_results)

        triples = []
        if show_graph:
            triples = await self.context_retriever.get_triples(question)
            self.visualizer.render(triples, title=f"Context for: {question[:50]}")

        return QAResult(
            question=question,
            predicted_scope=classification.scope,
            search_type_used=search_type.value,
            answer=answer,
            context_triples=triples,
            classify_time_sec=round(classify_elapsed, 2),
            answer_time_sec=round(answer_elapsed, 2),
            classify_tokens=classify_usage["total_tokens"],
            answer_tokens=answer_usage["total_tokens"],
        )

In [23]:
question_router    = QuestionRouter(tracker=tracker)
context_retriever  = GraphContextRetriever(tracker=tracker, top_k=5)
visualizer         = GraphVisualizer()

qa_engine = CogneeQAEngine(
    router=question_router,
    context_retriever=context_retriever,
    visualizer=visualizer,
    tracker=tracker,
    system_prompt=EXERCISE_QA_SYSTEM_PROMPT,
)

test_questions = [
    "what excerise are there",                          # local
]

for q in test_questions:
    result = await qa_engine.ask(q)
    print(f"[{result.predicted_scope:>11}] -> {result.search_type_used:<22} | {q}")
    print(f"   answer: {result.answer[:150]}")
    print()


2026-06-22T09:40:19.129290 [info     ] Vector collection retrieval completed: Retrieved distances from 10 collections in 0.74s [cognee.shared.logging_utils]

2026-06-22T09:40:19.131177 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-22T09:40:19.177530 [info     ] ID-filtered retrieval: 8 nodes and 7 edges in 0.05s [Neo4jAdapter]

2026-06-22T09:40:19.178731 [info     ] Graph projection completed: 8 nodes, 7 edges in 0.00s [CogneeGraph]

2026-06-22T09:40:19.185447 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 8, 'connection_count': 7}

2026-06-22T09:40:27.760041 [info     ] Vector collection retrieval completed: Retrieved distances from 10 collections in 0.85s [cognee.shared.logging_utils]

2026-06-22T09:40:27.761052 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-22T09:40:27.810430 [info     ] ID-filtered retrieval: 8 nodes and 7 edges in 0.05s [Neo4jAdapter]

2026-06-22T09:

Context for: what excerise are there -> saved to D:\longmemory\LLM-Memory\.artifacts\graphs\graph_3054464226424759127.html


[     global] -> GRAPH_SUMMARY_COMPLETION | what excerise are there
   answer: the jack jump is a beginner-level exercise that requires body weight. it targets the cardiovascular system and is considered cardio. the primary and s



## Step 3 — Golden Q&A evaluation across search types

`QAEvaluator` runs a fixed golden test set through several `SearchType`s, scores
pass/fail against expected terms, and exports a formatted Excel comparison report.

In [ ]:
QA_TESTS = [
    # ---- your existing questions (Q01 corrected; Q02/Q04/Q05 unchanged & verified) ----
    {
        "id": "Q01",
        "question": "Tell me the instructions for sit up",
        "expected_scope": "instruction",
        "must_contain": ["knees bent", "abs", "curling forward"],
        "must_not_contain": ["dumbbell", "barbell", "machine"],
    },
    {
        "id": "Q02",
        "question": "I want to improve my upper body strength, what exercises should I focus on?",
        "expected_scope": "local",
        "must_contain": ["bodyweight"],
        "must_not_contain": ["calves", "forearms", "wrist"],
    },
    {
        "id": "Q03",
        "question": "i am doing cardio, upper body training and legs very week review my exercise plan i am 50 yrs old and i weigh 100 kgs",
        "expected_scope": "multi_hop",
        "must_contain": ["Cardio", "Upper body", "Legs"],
        "must_not_contain": ["wrist"],
        # Soft check: this is a coaching-synthesis question, not tied to one record —
        # a fail here tells you about generation quality, not dataset grounding.
    },
    {
        "id": "Q04",
        "question": "Tell me how to do a proper squat to build stronger legs and glutes",
        "expected_scope": "instruction",
        "must_contain": ["glutes", "knees", "feet shoulder-width"],
        "must_not_contain": ["pectorals", "forearms", "wrist"],
    },
    {
        "id": "Q05",
        "question": "I want to tone my stomach and core, what exercises should I do?",
        "expected_scope": "multi_hop",
        "must_contain": ["leg", "crunch"],
        "must_not_contain": ["pectorals", "biceps", "calves"],
    },

    # ---- new realistic trainer-style questions, verified against the dataset ----
    {
        "id": "Q06",
        "question": "I'm new to working out, what's a good beginner exercise for my abs?",
        "expected_scope": "local",
        "must_contain": ["abs", "beginner"],
        "must_not_contain": ["dumbbell", "barbell", "weighted"],
    },
    {
        "id": "Q07",
        "question": "I have a kettlebell at home, what's a good shoulder exercise I can do with it?",
        "expected_scope": "local",
        "must_contain": ["kettlebell", "shoulders"],
        "must_not_contain": ["beginner"],
    },
    {
        "id": "Q08",
        "question": "What's a good pull-up variation for building my back without any weights?",
        "expected_scope": "multi_hop",
        "must_contain": ["lats", "body weight"],
        "must_not_contain": ["dumbbell", "barbell"],
    },
    {
        "id": "Q09",
        "question": "Can you suggest a dumbbell shoulder exercise I can do at home?",
        "expected_scope": "local",
        "must_contain": ["dumbbell", "delts"],
        "must_not_contain": ["barbell", "cable"],
    },
    {
        "id": "Q10",
        "question": "What's a good resistance band exercise for my biceps?",
        "expected_scope": "local",
        "must_contain": ["band", "biceps"],
        "must_not_contain": ["dumbbell", "barbell"],
    },
    {
        "id": "Q11",
        "question": "I want to strengthen my lower back, what exercises help with that?",
        "expected_scope": "local",
        "must_contain": ["pelvic"],
        "must_not_contain": ["dumbbell", "barbell"],
    },
    {
        "id": "Q12",
        "question": "I want to do chest exercises using a kettlebell, what do you suggest?",
        "expected_scope": "multi_hop",
        "must_contain": ["kettlebell", "pectorals"],
        "must_not_contain": ["barbell", "dumbbell"],
    },
    {
        "id": "Q13",
        "question": "What's a beginner-friendly back exercise I can do without any equipment?",
        "expected_scope": "multi_hop",
        "must_contain": ["beginner", "bodyweight"],
        "must_not_contain": ["dumbbell", "barbell", "cable"],
    },
    {
        "id": "Q14",
        "question": "I want to build bigger shoulders, what exercises should I try?",
        "expected_scope": "local",
        "must_contain": ["shrug"],
        "must_not_contain": ["quadriceps", "calves", "glutes"],
    },
    {
        "id": "Q15",
        "question": "Can you give me an exercise to improve my cardio endurance without any equipment?",
        "expected_scope": "multi_hop",
        "must_contain": ["cardio", "bodyweight"],
        "must_not_contain": ["dumbbell", "barbell"],
    },
    {
        "id": "Q16",
        "question": "What's a good forearm exercise I can do using a cable machine?",
        "expected_scope": "multi_hop",
        "must_contain": ["cable", "forearms"],
        "must_not_contain": ["dumbbell", "barbell"],
    },
    {
        "id": "Q17",
        "question": "I want to work on my obliques, what core exercise should I do?",
        "expected_scope": "local",
        "must_contain": ["obliques"],
        "must_not_contain": ["pectorals", "biceps"],
    },
    {
        "id": "Q18",
        "question": "What's an advanced or hard-level exercise I could try once I've built up some strength?",
        "expected_scope": "local",
        "must_contain": ["hard"],
        "must_not_contain": ["beginner"],
    },
    {
        "id": "Q19",
        "question": "How many different exercise categories does this program cover?",
        "expected_scope": "global",
        "must_contain": ["chest", "back"],
        "must_not_contain": ["swimming", "yoga"],
    },
    {
        "id": "Q20",
        "question": "I want a lower-body workout that combines glutes and hamstrings, what should I include?",
        "expected_scope": "multi_hop",
        "must_contain": ["glutes", "hamstrings"],
        "must_not_contain": ["pectorals", "biceps"],
    },
]

print(f"Golden Q&A test cases: {len(QA_TESTS)}")
import collections
print("By expected scope:", collections.Counter(tc["expected_scope"] for tc in QA_TESTS))

Golden Q&A test cases: 20
By expected scope: Counter({'multi_hop': 11, 'local': 6, 'instruction': 2, 'global': 1})


In [20]:
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


class QAEvaluator:
    """Runs a golden Q&A test set across multiple SearchTypes, scores pass/fail
    against expected terms, and exports an Excel comparison report."""

    def __init__(self, tracker: TokenUsageTracker, system_prompt: Optional[str] = None):
        self.tracker = tracker
        self.system_prompt = system_prompt
        self.results_df: Optional[pd.DataFrame] = None
        self.comparison_df: Optional[pd.DataFrame] = None

    @staticmethod
    def _check(answer: str, must_contain: list, must_not_contain: list):
        missing = [t for t in must_contain if t.lower() not in answer]
        present = [t for t in must_not_contain if t.lower() in answer]
        return (not missing and not present), missing, present

    async def run(self, qa_tests: list[dict], router: "QuestionRouter") -> pd.DataFrame:
        rows = []
        for tc in qa_tests:
            print(f"Running {tc['id']} [{tc['expected_scope']}]: {tc['question'][:55]}\u2026")

            t0 = time.perf_counter()
            classification = await router.classify(tc["question"])
            classify_elapsed = time.perf_counter() - t0
            search_type = router.search_type_for(classification.scope)

            phase_name = f"{tc['id']}__{search_type.value}"
            t1 = time.perf_counter()
            with self.tracker.phase(phase_name):
                try:
                    raw_results = await search(query_text=tc["question"], query_type=search_type,
                                                system_prompt=self.system_prompt)
                    answer = answer_to_str(raw_results)
                    api_error = None
                except Exception as e:
                    answer, api_error = "", str(e)
            elapsed = time.perf_counter() - t1
            usage = self.tracker.summarize(phase_name)

            if api_error:
                passed, missing, false_hits = False, ["API ERROR"], []
            else:
                passed, missing, false_hits = self._check(answer, tc["must_contain"], tc["must_not_contain"])

            scope_correct = classification.scope == tc["expected_scope"]

            rows.append({
                "id": tc["id"], "expected_scope": tc["expected_scope"], "predicted_scope": classification.scope,
                "scope_correct": "\u2705" if scope_correct else "\u274c", "search_type_used": search_type.value,
                "question": tc["question"],
                "expected_contains": ", ".join(tc["must_contain"]),
                "expected_not_contains": ", ".join(tc["must_not_contain"]),
                "answer": answer, "result": "\u2705 PASS" if passed else "\u274c FAIL",
                "missing_terms": ", ".join(missing), "unexpected_terms": ", ".join(false_hits),
                "api_error": api_error or "", "classify_time_sec": round(classify_elapsed, 2),
                "time_sec": round(elapsed, 2), "llm_calls": usage["calls"],
                "prompt_tokens": usage["prompt_tokens"], "completion_tokens": usage["completion_tokens"],
                "total_tokens": usage["total_tokens"],
            })

        self.results_df = pd.DataFrame(rows)
        self.comparison_df = self._build_comparison(self.results_df)
        return self.results_df

    @staticmethod
    def _build_comparison(df: pd.DataFrame) -> pd.DataFrame:
        comp = df.groupby(["expected_scope", "predicted_scope", "search_type_used"]).agg(
            pass_count=("result", lambda s: (s == "\u2705 PASS").sum()),
            total_count=("result", "count"),
            scope_accuracy=("scope_correct", lambda s: (s == "\u2705").mean() * 100),
            avg_time_sec=("time_sec", "mean"),
            avg_total_tokens=("total_tokens", "mean"),
        ).round(2)
        comp["pass_rate"] = (comp["pass_count"] / comp["total_count"] * 100).round(1)
        return comp

    def print_summary(self):
        df = self.results_df
        passed = (df["result"] == "\u2705 PASS").sum()
        print(f"Overall: {passed}/{len(df)} passed \u2014 {len(df) - passed} failed\n")
        display(df[["id", "expected_scope", "predicted_scope", "scope_correct", "search_type_used",
                    "question", "result", "time_sec", "llm_calls", "prompt_tokens",
                    "completion_tokens", "total_tokens"]])
        print("\n=== Comparison by expected/predicted scope + search type ===")
        display(self.comparison_df)

    def print_failures(self):
        failures = self.results_df[self.results_df["result"] == "\u274c FAIL"]
        if failures.empty:
            print("All tests passed \u2014 nothing to debug.")
            return
        print(f"{len(failures)} test(s) failed:\n")
        for _, row in failures.iterrows():
            print(f"  {row['id']} [{row['search_type_used']}]  {row['question']}")
            if row["missing_terms"]:
                print(f"    Missing:    {row['missing_terms']}")
            if row["unexpected_terms"]:
                print(f"    Unexpected: {row['unexpected_terms']}")
            if row["api_error"]:
                print(f"    API error:  {row['api_error']}")
            print(f"    Answer:     {row['answer'][:200]}")
            print()

    def export_excel(self, path: str = "./qa_search_type_comparison.xlsx") -> str:
        columns = ["id", "expected_scope", "predicted_scope", "scope_correct", "search_type_used",
        "question", "expected_contains", "expected_not_contains", "answer", "result",
        "missing_terms", "unexpected_terms", "api_error", "classify_time_sec", "time_sec",
        "llm_calls", "prompt_tokens", "completion_tokens", "total_tokens"]
        headers = ["ID", "Expected Scope", "Predicted Scope", "Scope Correct", "Search Type Used",
                "Question", "Expected To Contain", "Expected NOT To Contain", "Answer", "Result",
                "Missing Terms", "Unexpected Terms", "API Error", "Classify Time (s)", "Answer Time (s)",
                "LLM Calls", "Prompt Tokens", "Completion Tokens", "Total Tokens"]

        results_df = self.results_df[columns].copy()
        results_df.columns = headers

        comparison_df = self.comparison_df.reset_index()
        comparison_df = comparison_df[["expected_scope", "predicted_scope", "search_type_used",
                                        "pass_count", "total_count", "pass_rate",
                                        "scope_accuracy", "avg_time_sec", "avg_total_tokens"]]
        comparison_df.columns = ["Expected Scope", "Predicted Scope", "Search Type Used",
                                "Passed", "Total", "Pass Rate (%)", "Scope Accuracy (%)",
                                "Avg Time (s)", "Avg Total Tokens"]

        excel_path = str(pathlib.Path(path).resolve())
        with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
            results_df.to_excel(writer, sheet_name="Results", index=False)
            comparison_df.to_excel(writer, sheet_name="Comparison", index=False)

            header_fill = PatternFill("solid", start_color="1F4E78")
            pass_fill   = PatternFill("solid", start_color="C6EFCE")
            fail_fill   = PatternFill("solid", start_color="FFC7CE")

            ws = writer.sheets["Results"]
            for cell in ws[1]:
                cell.font = Font(name="Arial", bold=True, color="FFFFFF")
                cell.fill = header_fill
                cell.alignment = Alignment(horizontal="center", vertical="center")
            result_col = headers.index("Result") + 1
            for r in range(2, ws.max_row + 1):
                is_pass = "PASS" in str(ws.cell(row=r, column=result_col).value)
                for c in range(1, len(headers) + 1):
                    cell = ws.cell(row=r, column=c)
                    cell.font = Font(name="Arial")
                    cell.alignment = Alignment(vertical="top", wrap_text=True)
                ws.cell(row=r, column=result_col).fill = pass_fill if is_pass else fail_fill
            widths = {"ID": 6, "Expected Scope": 14, "Predicted Scope": 14, "Scope Correct": 12,
                    "Search Type Used": 24, "Question": 38, "Expected To Contain": 24,
                    "Expected NOT To Contain": 24, "Answer": 55, "Result": 10, "Missing Terms": 18,
                    "Unexpected Terms": 18, "API Error": 18, "Classify Time (s)": 14, "Answer Time (s)": 14,
                    "LLM Calls": 10, "Prompt Tokens": 12, "Completion Tokens": 14, "Total Tokens": 12}
            for i, h in enumerate(headers, start=1):
                ws.column_dimensions[get_column_letter(i)].width = widths.get(h, 15)
            ws.freeze_panes = "A2"

            ws2 = writer.sheets["Comparison"]
            for cell in ws2[1]:
                cell.font = Font(name="Arial", bold=True, color="FFFFFF")
                cell.fill = header_fill
                cell.alignment = Alignment(horizontal="center", vertical="center")
            for r in range(2, ws2.max_row + 1):
                for c in range(1, len(comparison_df.columns) + 1):
                    cell = ws2.cell(row=r, column=c)
                    cell.font = Font(name="Arial")
                    cell.alignment = Alignment(horizontal="center")
            for i in range(1, len(comparison_df.columns) + 1):
                ws2.column_dimensions[get_column_letter(i)].width = 18
            ws2.freeze_panes = "A2"

        print(f"Excel report saved to: {excel_path}")
        return excel_path

In [21]:
evaluator = QAEvaluator(tracker=tracker, system_prompt=EXERCISE_QA_SYSTEM_PROMPT)
await evaluator.run(QA_TESTS, router=question_router)   # reuse the QuestionRouter instance from qa-engine-demo
evaluator.print_summary()

Running Q01 [instruction]: Tell me the instructions for sit up…
Running Q02 [local]: I want to improve my upper body strength, what exercise…



2026-06-22T04:34:09.765260 [info     ] Vector collection retrieval completed: Retrieved distances from 10 collections in 2.92s [cognee.shared.logging_utils]

2026-06-22T04:34:09.772384 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-22T04:34:11.718227 [info     ] ID-filtered retrieval: 617 nodes and 3480 edges in 1.94s [Neo4jAdapter]

2026-06-22T04:34:11.878603 [info     ] Graph projection completed: 617 nodes, 3480 edges in 0.15s [CogneeGraph]

2026-06-22T04:34:11.967760 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 13, 'connection_count': 10}


Running Q03 [multi_hop]: i am doing cardio, upper body training and legs very we…



2026-06-22T04:34:29.319127 [info     ] Vector collection retrieval completed: Retrieved distances from 10 collections in 11.18s [cognee.shared.logging_utils]

2026-06-22T04:34:29.449032 [info     ] Retrieving full graph.         [CogneeGraph]

2026-06-22T04:34:31.048490 [info     ] Retrieved 617 nodes and 3480 edges in 1.60 seconds [Neo4jAdapter]

2026-06-22T04:34:31.168001 [info     ] Graph projection completed: 617 nodes, 3480 edges in 0.12s [CogneeGraph]

2026-06-22T04:34:31.275602 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 12, 'connection_count': 10}

2026-06-22T04:34:32.753370 [info     ] Chain-of-thought: generated completions for 1 queries [cognee.shared.logging_utils]

2026-06-22T04:34:35.526023 [info     ] Chain-of-thought: follow-up questions: ['Could you please provide details about your current cardio and upper body training routines?'] [cognee.shared.logging_utils]

2026-06-22T04:34:41.091563 [info     ] Vector collect

Running Q04 [instruction]: Tell me how to do a proper squat to build stronger legs…
Running Q05 [local]: I want to tone my stomach and core, what exercises shou…


...


2026-06-22T04:41:06.560497 [info     ] Vector collection retrieval completed: Retrieved distances from 10 collections in 5.18s [cognee.shared.logging_utils]

2026-06-22T04:41:06.683205 [info     ] Retrieving full graph.         [CogneeGraph]

2026-06-22T04:41:07.945390 [info     ] Retrieved 617 nodes and 3480 edges in 1.26 seconds [Neo4jAdapter]

2026-06-22T04:41:08.319615 [info     ] Graph projection completed: 617 nodes, 3480 edges in 0.35s [CogneeGraph]

2026-06-22T04:41:08.419191 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 10, 'connection_count': 10}

2026-06-22T04:41:09.801895 [info     ] Chain-of-thought: generated completions for 1 queries [cognee.shared.logging_utils]

2026-06-22T04:41:13.154018 [info     ] Chain-of-thought: follow-up questions: ['What other exercises target glutes and hamstrings?'] [cognee.shared.logging_utils]

2026-06-22T04:41:19.798606 [info     ] Vector collection retrieval completed: Retrieved distance

Overall: 12/20 passed — 8 failed



,id,expected_scope,predicted_scope,scope_correct,search_type_used,question,result,time_sec,llm_calls,prompt_tokens,completion_tokens,total_tokens
0,Q01,instruction,instruction,✅,TRIPLET_COMPLETION,Tell me the instructions for sit up,✅ PASS,3.80,2,1704,109,1813
1,Q02,local,global,❌,GRAPH_SUMMARY_COMPLETION,"I want to improve my upper body strength, what...",❌ FAIL,8.85,3,1829,167,1996
2,Q03,multi_hop,multi_hop,✅,GRAPH_COMPLETION_COT,"i am doing cardio, upper body training and leg...",✅ PASS,65.97,223,22756,1562,24318
3,Q04,instruction,instruction,✅,TRIPLET_COMPLETION,Tell me how to do a proper squat to build stro...,✅ PASS,2.88,2,1822,129,1951
4,Q05,local,multi_hop,❌,GRAPH_COMPLETION_COT,"I want to tone my stomach and core, what exerc...",✅ PASS,57.70,223,17932,952,18884
5,Q06,multi_hop,local,❌,GRAPH_COMPLETION,"I'm new to working out, what's a good beginner...",✅ PASS,7.73,2,1151,34,1185
6,Q07,multi_hop,local,❌,GRAPH_COMPLETION,"I have a kettlebell at home, what's a good sho...",❌ FAIL,5.62,2,1140,63,1203
7,Q08,multi_hop,local,❌,GRAPH_COMPLETION,What's a good pull-up variation for building m...,❌ FAIL,5.96,2,1136,84,1220
8,Q09,multi_hop,local,❌,GRAPH_COMPLETION,Can you suggest a dumbbell shoulder exercise I...,✅ PASS,5.74,2,1141,71,1212
9,Q10,multi_hop,local,❌,GRAPH_COMPLETION,What's a good resistance band exercise for my ...,✅ PASS,6.10,2,1149,29,1178



=== Comparison by expected/predicted scope + search type ===


pass_count  \
expected_scope predicted_scope search_type_used                       
global         global          GRAPH_SUMMARY_COMPLETION           0   
instruction    instruction     TRIPLET_COMPLETION                 2   
local          global          GRAPH_SUMMARY_COMPLETION           0   
               local           GRAPH_COMPLETION                   2   
               multi_hop       GRAPH_COMPLETION_COT               1   
multi_hop      local           GRAPH_COMPLETION                   4   
               multi_hop       GRAPH_COMPLETION_COT               3   

                                                         total_count  \
expected_scope predicted_scope search_type_used                        
global         global          GRAPH_SUMMARY_COMPLETION            1   
instruction    instruction     TRIPLET_COMPLETION                  2   
local          global          GRAPH_SUMMARY_COMPLETION            2   
               local           GRAPH_COMPLETION                    3   
               multi_hop       GRAPH_COMPLETION_COT                1   
multi_hop      local           GRAPH_COMPLETION                    6   
               multi_hop       GRAPH_COMPLETION_COT                5   

                                                         scope_accuracy  \
expected_scope predicted_scope search_type_used                           
global         global          GRAPH_SUMMARY_COMPLETION           100.0   
instruction    instruction     TRIPLET_COMPLETION                 100.0   
local          global          GRAPH_SUMMARY_COMPLETION             0.0   
               local           GRAPH_COMPLETION                   100.0   
               multi_hop       GRAPH_COMPLETION_COT                 0.0   
multi_hop      local           GRAPH_COMPLETION                     0.0   
               multi_hop       GRAPH_COMPLETION_COT               100.0   

                                                         avg_time_sec  \
expected_scope predicted_scope search_type_used                         
global         global          GRAPH_SUMMARY_COMPLETION          9.95   
instruction    instruction     TRIPLET_COMPLETION                3.34   
local          global          GRAPH_SUMMARY_COMPLETION          7.98   
               local           GRAPH_COMPLETION                  7.44   
               multi_hop       GRAPH_COMPLETION_COT             57.70   
multi_hop      local           GRAPH_COMPLETION                  6.14   
               multi_hop       GRAPH_COMPLETION_COT             59.12   

                                                         avg_total_tokens  \
expected_scope predicted_scope search_type_used                             
global         global          GRAPH_SUMMARY_COMPLETION           2050.00   
instruction    instruction     TRIPLET_COMPLETION                 1882.00   
local          global          GRAPH_SUMMARY_COMPLETION           1879.00   
               local           GRAPH_COMPLETION                   1216.33   
               multi_hop       GRAPH_COMPLETION_COT              18884.00   
multi_hop      local           GRAPH_COMPLETION                   1193.50   
               multi_hop       GRAPH_COMPLETION_COT              20762.60   

                                                         pass_rate  
expected_scope predicted_scope search_type_used                     
global         global          GRAPH_SUMMARY_COMPLETION        0.0  
instruction    instruction     TRIPLET_COMPLETION            100.0  
local          global          GRAPH_SUMMARY_COMPLETION        0.0  
               local           GRAPH_COMPLETION               66.7  
               multi_hop       GRAPH_COMPLETION_COT          100.0  
multi_hop      local           GRAPH_COMPLETION               66.7  
               multi_hop       GRAPH_COMPLETION_COT           60.0

In [22]:
evaluator.print_failures()

8 test(s) failed:

  Q02 [GRAPH_SUMMARY_COMPLETION]  I want to improve my upper body strength, what exercises should I focus on?
    Missing:    chest, pectorals
    Answer:     to improve your upper body strength, you can focus on exercises that target the upper back, such as: bodyweight standing row, lever high row, lever t-bar reverse grip row, inverted row on bench, bodyw

  Q07 [GRAPH_COMPLETION]  I have a kettlebell at home, what's a good shoulder exercise I can do with it?
    Missing:    delts
    Answer:     the kettlebell double windmill, kettlebell sumo high pull, kettlebell windmill, kettlebell double alternating hang clean, and kettlebell one arm floor press all work the shoulders and require a kettle

  Q08 [GRAPH_COMPLETION]  What's a good pull-up variation for building my back without any weights?
    Missing:    lats, body weight
    Answer:     i don't have any bodyweight pull-up variations listed in the provided context. the exercises found (rocky pull-up pulldown, w

In [23]:
debug_df = tracker.as_dataframe()
print(f"Total captured calls: {len(debug_df)}")
if not debug_df.empty:
    display(debug_df)
else:
    print("No calls captured \u2014 the monkeypatch isn't intercepting cognee's LLM calls.")

Total captured calls: 6394


,phase,model,call_type,prompt_tokens,completion_tokens,total_tokens,elapsed_sec
0,graph_and_embedding_creation,gemini/gemini-embedding-001,embedding,2,0,2,1.533868
1,graph_and_embedding_creation,gemini/gemini-embedding-001,embedding,2,0,2,1.741207
2,graph_and_embedding_creation,gemini/gemini-embedding-001,embedding,2,0,2,1.488077
3,graph_and_embedding_creation,gemini/gemini-embedding-001,embedding,76,0,76,1.761521
4,graph_and_embedding_creation,gemini/gemini-embedding-001,embedding,1,0,1,1.599969
...,...,...,...,...,...,...,...
6389,Q20__GRAPH_COMPLETION_COT,gemini/gemini-embedding-001,embedding,136,0,136,0.944276
6390,Q20__GRAPH_COMPLETION_COT,gemini/gemini-embedding-001,embedding,136,0,136,0.984279
6391,Q20__GRAPH_COMPLETION_COT,gemini/gemini-embedding-001,embedding,136,0,136,0.994535
6392,Q20__GRAPH_COMPLETION_COT,gemini/gemini-embedding-001,embedding,136,0,136,1.199223


In [24]:
evaluator.export_excel()

Excel report saved to: D:\longmemory\long-memory-rag\qa_search_type_comparison.xlsx


'D:\\longmemory\\long-memory-rag\\qa_search_type_comparison.xlsx'